# Eye-state classifier branch (deep learning)

Same job as `EAR_Detection.ipynb` (PERCLOS, eye-closure time, blink rate, confidence), but the open/closed decision comes from a **neural network that looks at the eye picture** instead of landmark geometry.
So the two branches fail in different situations, which is what makes fusing them useful.

```
frame -> face detector (YuNet: box + eye centres) -> two eye crops -> CNN -> p(closed) per eye
      -> state OPEN / CLOSED / UNKNOWN -> rolling 30 s window -> PERCLOS, closure time, blinks, score, confidence
```
The notebook goes in order: **1. image, 2. video file, 3. live camera**. The code lives in the package (`drowsiness/video/cnn_branch.py`, the window statistics in `drowsiness/common/windowing.py`, the same rules as the EAR notebook), so the fusion uses exactly what is shown here.
From the terminal: `python -m drowsiness.video.cnn_branch --source 0` (camera) or `--source data/test_vid_1.mov`.

## 0. Setup

In [1]:
# Run once in the project root, then pick the .venv kernel for this notebook:
# python3 -m venv .venv && .venv/bin/pip install -r requirements.txt
# python3 download_models.py          # YuNet face detector (and the old model, kept for comparison)
# python3 train_eye_model.py train small 80 14   # trains our eye model (about 10 minutes) -> models/eye_state_small_80.onnx
import json
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from drowsiness.common.windowing import MICROSLEEP_DURATION, SLEEP_DURATION, UNRESPONSIVE_DURATION
from drowsiness.video.cnn_branch import CLOSED_ON, EYE_MODEL, FACE_MODEL, CnnBranch, crop_eye
from drowsiness.video.hud import draw_hud
from drowsiness.video.runner import run_video, summary

IMAGE_PATH = "data/test2.jpg"  # your own test image (not in git)
VIDEO_PATH = "data/test_vid_1.mov"  # your own test video (not in git)

for path in (FACE_MODEL, EYE_MODEL):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: see the commands at the top of this notebook")

## 1. Which model, and why

The old OpenVINO model was not used; four candidates were measured on the same eye images (script `train_eye_model.py compare`, full table in `docs/eye_model_comparison.md`).
Two test sets: **MRL** = 6000 infrared eye images of 8 people the model never trained on, **webcam** = 206 colour crops from your video (one person only, so just a sanity check).
`false open` = a closed eye called open (the dangerous mistake).

In [2]:
results = json.load(open("docs/eye_model_comparison.json"))
rows = []
for name, r in results.items():
    if name.endswith("_ms"):
        continue
    rows.append(
        {
            "model": name,
            "params": r["params"],
            "licence": r["licence"],
            "MRL accuracy %": 100 * r["mrl"]["accuracy"],
            "MRL false open %": 100 * r["mrl"]["false_open_rate"],
            "MRL false closed %": 100 * r["mrl"]["false_closed_rate"],
            "webcam accuracy %": 100 * r["webcam"]["accuracy"],
            "CPU ms per eye": r["ms_per_eye_cpu"],
        }
    )
pd.DataFrame(rows).round(1).set_index("model")

,params,licence,MRL accuracy %,MRL false open %,MRL false closed %,webcam accuracy %,CPU ms per eye
model,,,,,,,
OpenVINO open-closed-eye-0001 (old),0.05 M,Apache-2.0,57.6,100.0,0.1,68.9,0.1
"MobileNetV2 (Michal M., HF)",4.4 M,CC BY-NC-ND 4.0,52.4,0.0,82.6,90.8,6.6
"ViT-base (dima806, HF)",86 M,Apache-2.0,99.2,1.3,0.5,100.0,48.1
MobileNetV3 fine-tuned (ours),1.5 M,own weights; torchvision BSD-3,97.2,1.1,4.0,100.0,0.8


## 2. Face detector, eye crops, classifier

* **Face + eye centres: YuNet** (OpenCV, MIT licence). It returns the face box and both eye centres, finds the face in more frames than the Haar detector, and does **not** need dlib landmarks. So if the EAR branch loses its landmarks, this branch still works.
* **Eye crop:** a square around each eye centre, side = 0.65 x the distance between the eyes. That framing (eye + brow) is what the training images look like.
* **Classifier:** MobileNetV3 fine-tuned on the MRL Eye dataset (84,898 eye images, 37 people, split by person), with strong augmentation (light, blur, noise, rotation, mirror). Input: grey 80x80 px eye, output: probability the eye is closed. Runs in about 1 ms with ONNX Runtime.

## 3. Confidence: how much can this frame be trusted?

Confidence (0-1) is the **minimum** of eight simple factors, so any one problem lowers it (a chain is as strong as its weakest link). No face gives 0.

| factor | drops when |
|---|---|
| `face` | the face detector is unsure it is a face |
| `size` | the eyes are only a few pixels wide (far away, low resolution) |
| `pose` | the head is turned (nose off centre) or strongly tilted |
| `inside` | an eye crop falls partly outside the image |
| `light` | the eyes are too dark or over-exposed |
| `agree` | the two eyes disagree strongly (glare on glasses, hand or hair over one eye) |
| `certain` | the network is undecided (probability near 0.5): no hard verdict from an unclear eye |
| `stable` | the eye positions jump between frames (tracking is shaky) |

Frames below `MIN_CONF` become `UNKNOWN`: they are never counted as open or closed, and PERCLOS ignores them.

`input_confidence` and the rest of the branch are in `drowsiness/video/cnn_branch.py`: face detector (YuNet), eye crops, the network, the eight confidence factors, the steadier verdict (p(closed) averaged over 3 frames; an eye that is CLOSED needs p below 0.4 to count as OPEN again, which stops flicker), and the window statistics.

## 4. Step 1: a single image

In [3]:
# This cell shows a face: its picture is not kept in the saved notebook (tests/test_notebooks.py checks that before a commit).
branch = CnnBranch()
branch.start()

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

branch.update(0.0, image)
res = branch.last
if res["face"] is None:
    print("No face found in the image")
else:
    print(f"state {res['state']}   p(closed): right eye {res['p_right']:.3f}, left eye {res['p_left']:.3f}")
    print(f"confidence {res['confidence']:.2f}   factors { ({k: round(v, 2) for k, v in res['factors'].items()}) }")
    print(f"face detector {res['ms_face']:.1f} ms, classifier {res['ms_cnn']:.1f} ms")

    shown = image.copy()
    draw_hud(shown, branch)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 1, 1]})
    axes[0].imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    for ax, centre, p, title in zip(axes[1:], res["eyes"], (res["p_right"], res["p_left"]), ("right eye", "left eye")):
        crop, _ = crop_eye(image, centre, res["side"])
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title}: p(closed) {p:.2f}")
    for ax in axes:
        ax.axis("off")
    plt.show()

state OPEN   p(closed): right eye 0.000, left eye 0.000
confidence 1.00   factors {'face': 1.0, 'size': 1.0, 'pose': 1.0, 'inside': 1.0, 'light': 1.0, 'agree': 1.0, 'certain': 1.0, 'stable': 1.0}
face detector 17.2 ms, classifier 5.8 ms


## 5. Steps 2 and 3: video file and live camera (one function)

`run_video(branch, source)` takes a **video path, a camera index (0 = laptop camera) or a stream URL** and calculates everything **for every frame**, over a rolling 30 s window:
PERCLOS, current and longest eye-closure time, blink count and rate, drowsiness score, and the **confidence** of the channel (mean frame confidence over the last 3 s, capped by the last half second, so it falls at once when the face or eyes are lost).

* `show="notebook"` shows frames inline, `"window"` opens an OpenCV window (press `q`), `None` shows nothing.
* `save_to="runs/x.mp4"` writes the annotated video, `log_csv="runs/x.csv"` writes one row per frame.
* No calibration is needed: the network decides from the picture, not from a person-specific threshold.
* **macOS camera:** the app that runs this notebook (Terminal, VS Code, Jupyter) needs permission under System Settings > Privacy & Security > Camera.

### 5a. Your video file

In [4]:
video = run_video(CnnBranch(), VIDEO_PATH, show=None, save_to="runs/dl_video_annotated.mp4", log_csv="runs/dl_video_log.csv")

print(len(video["t"]), "frames,", f"{video['t'][-1]:.1f} s")
print("speed:", summary(video))
print(f"        face detector {np.mean(video['ms_face']):.1f} ms, classifier {np.mean(video['ms_cnn']):.1f} ms per frame")
final = video["output"]
print(f"\nscore {final['score']:.2f} (0 = alert, 1 = drowsy)    channel confidence {final['confidence']:.2f}")
for key, value in final["raw"].items():
    print(f"  {key}: {value}")
print("\nannotated video: runs/dl_video_annotated.mp4   per-frame log: runs/dl_video_log.csv")

Stream ended or camera stopped delivering frames.
641 frames, 21.6 s
speed: 641 frames in 21.6 s; processing 16.1 ms per frame (= 62 FPS possible, 95th percentile 20.3 ms)
        face detector 11.9 ms, classifier 3.5 ms per frame

score 0.75 (0 = alert, 1 = drowsy)    channel confidence 0.39
  eyes_closed: False
  current_closure_s: 0.0
  closure_level: NONE
  longest_closure_s: 3.166666666666668
  recent_longest_closure_s: 3.166666666666668
  blink_count: 7
  blink_rate_per_min: 19.476002782286113
  perclos: 0.2400793650793651
  face_found_share: 0.7862714508580343
  window_confidence: 0.7581172408578742
  window_s: 21.565

annotated video: runs/dl_video_annotated.mp4   per-frame log: runs/dl_video_log.csv


The values below are calculated **for every frame** over the rolling 30 s window (blue = this notebook's CNN branch). Grey bands: frames with no usable face (`UNKNOWN`).

In [5]:
t = video["t"]
unknown = video["state"] == "UNKNOWN"
fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
panels = [
    ("p(closed), both eyes", video["p_closed"], (-0.05, 1.05)),
    ("current eye-closure time (s)", video["closure_s"], None),
    ("PERCLOS (rolling window)", video["perclos"], (-0.05, 1.05)),
    ("drowsiness score (0 alert, 1 drowsy)", video["score"], (-0.05, 1.05)),
    ("confidence: frame (light) and channel (dark)", video["channel_confidence"], (-0.05, 1.05)),
]
for ax, (title, values, ylim) in zip(axes, panels):
    ax.plot(t, values, color="tab:blue")
    ax.set_ylabel(title, fontsize=8)
    ax.grid(alpha=0.3)
    if ylim:
        ax.set_ylim(*ylim)
    ax.fill_between(t, 0, 1, where=unknown, transform=ax.get_xaxis_transform(), color="grey", alpha=0.2)
axes[0].axhline(CLOSED_ON, ls="--", color="k", lw=0.8)
axes[1].axhline(MICROSLEEP_DURATION, ls="--", color="orange", lw=0.8)  # 1 s: MICROSLEEP (Euro NCAP)
axes[1].axhline(SLEEP_DURATION, ls="--", color="red", lw=0.8)  # 3 s: SLEEP
axes[1].axhline(UNRESPONSIVE_DURATION, ls="--", color="darkred", lw=0.8)  # 6 s: UNRESPONSIVE
axes[4].plot(t, video["confidence"], color="tab:blue", alpha=0.3)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

### 5b. Live camera
Starts your laptop camera (`0`; try `1` for another camera, or a video path / stream URL). Interrupt the kernel (stop button) to end it, or set `max_seconds`.

In [ ]:
# This cell shows a face: its picture is not kept in the saved notebook (tests/test_notebooks.py checks that before a commit).
live = run_video(CnnBranch(), 0, show="notebook", log_csv="runs/dl_live_log.csv")
print(live["output"])